# 55 — GroupBy: split · apply · combine

Mở đầu nhóm trung cấp. Dữ liệu finlens ở **dạng long** — nhiều mã xếp chồng lên
nhau trong một frame — nên `groupby` không phải một kỹ thuật nâng cao mà là
**cách duy nhất** để làm gần như mọi việc cho đúng.

Notebook gồm:

1. `agg` · `transform` · `filter` — ba việc khác nhau, hay bị nhầm lẫn
2. ⚠️ **pandas 3.0 đổi mặc định `observed=` thành `True`** cho cột `category`
3. Bẫy ranh giới nhóm, đo bằng số
4. Hiệu năng: `category` làm `groupby` nhanh hơn bao nhiêu

In [1]:
import sys
import time
from pathlib import Path

GOC = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "finlens_examples").is_dir())
sys.path.insert(0, str(GOC))

import numpy as np
import pandas as pd

import finlens
from finlens_examples import hom_nay, lui_ngay

client = finlens.client()
HOM_NAY = hom_nay(client)

danh_muc = client.meta.symbols(exchange="HOSE", kind="stock")
gia = client.eod.stock.ohlcv(danh_muc["symbol"].tolist()[:150], start=lui_ngay(HOM_NAY, nam=2))
gia = gia.sort_values(["symbol", "date"]).reset_index(drop=True)

print(f"pandas {pd.__version__} · {gia['symbol'].nunique()} mã · {len(gia):,} dòng")

pandas 3.0.5 · 149 mã · 73,054 dòng


C:\Program Files\Python312\Lib\asyncio\base_events.py:1999: PartialDataWarning: 1 mã không lấy được: DMX. Chi tiết ở `df.attrs["finlens"]["failed"]`. Dùng `on_error="raise"` để biến thành ngoại lệ.
  handle._run()


## 1 · Ba việc khác nhau

Đây là bảng đáng học thuộc, vì chọn nhầm thì hình dạng kết quả sai ngay.

| Method | Vào | Ra | Dùng khi |
|---|---|---|---|
| `agg` | mỗi nhóm | **một dòng mỗi nhóm** | tóm tắt: tổng, trung bình, đếm |
| `transform` | mỗi nhóm | **cùng hình dạng đầu vào** | thêm cột so với chính nhóm |
| `filter` | mỗi nhóm | **tập con các nhóm** | giữ hoặc bỏ cả nhóm |
| `apply` | mỗi nhóm | tuỳ hàm | khi ba cái trên không đủ — và chậm |

In [2]:
theo_ma = gia.groupby("symbol", observed=True)

print(f"agg       → {theo_ma['close'].mean().shape}   một dòng mỗi mã")
print(f"transform → {theo_ma['close'].transform('mean').shape}   cùng hình dạng frame gốc")
print(f"filter    → {theo_ma.filter(lambda g: len(g) > 400).shape}   giữ cả nhóm hoặc bỏ cả nhóm")

agg       → (149,)   một dòng mỗi mã
transform → (73054,)   cùng hình dạng frame gốc
filter    → (72304, 7)   giữ cả nhóm hoặc bỏ cả nhóm


## 2 · `agg` — tóm tắt nhóm

**Named aggregation** là cách viết nên dùng: tên cột ra nằm bên trái, rõ ràng,
và không sinh ra `MultiIndex` cột.

In [3]:
tom_tat = gia.groupby("symbol", observed=True).agg(
    so_phien=("date", "count"),
    phien_dau=("date", "min"),
    phien_cuoi=("date", "max"),
    gia_cuoi=("close", "last"),
    gia_cao_nhat=("high", "max"),
    kl_bq=("volume", "mean"),
)
print(f"{len(tom_tat)} mã × {tom_tat.shape[1]} chỉ tiêu")
tom_tat.head(5).round({"gia_cuoi": 2, "gia_cao_nhat": 2, "kl_bq": 0})

149 mã × 6 chỉ tiêu


,so_phien,phien_dau,phien_cuoi,gia_cuoi,gia_cao_nhat,kl_bq
symbol,,,,,,
AAA,499,2024-08-12,2026-08-12,7.43,9.98,2033884.0
AAM,499,2024-08-12,2026-08-12,7.45,7.66,7141.0
AAN,58,2026-05-25,2026-08-12,15.45,17.34,214186.0
AAT,499,2024-08-12,2026-08-12,2.34,4.20,62682.0
ABR,499,2024-08-12,2026-08-12,13.50,16.49,2733.0


Cách cũ — truyền dict hoặc list — vẫn chạy nhưng sinh cột `MultiIndex`, và đó
là thứ bạn phải xử lý thêm ở bước sau:

In [4]:
kieu_cu = gia.groupby("symbol", observed=True).agg({"close": ["mean", "max"], "volume": "sum"})
print(f"agg kiểu cũ → cột là {type(kieu_cu.columns).__name__} {kieu_cu.columns.nlevels} tầng:")
print(f"  {kieu_cu.columns.tolist()}")
print()
print("Làm phẳng lại:")
phang = kieu_cu.copy()
phang.columns = ["_".join(c) for c in phang.columns]
print(f"  {phang.columns.tolist()}")

agg kiểu cũ → cột là MultiIndex 2 tầng:
  [('close', 'mean'), ('close', 'max'), ('volume', 'sum')]

Làm phẳng lại:
  ['close_mean', 'close_max', 'volume_sum']


### Hàm tự viết trong `agg`

`agg` nhận cả hàm — nhưng hàm nhận vào **một `Series` của nhóm**, không phải
một frame.

In [5]:
def bien_do_trung_binh(s: pd.Series) -> float:
    """Độ lệch chuẩn chia trung bình — hệ số biến thiên."""
    return s.std() / s.mean() * 100


bien_thien = (
    gia.groupby("symbol", observed=True)
    .agg(
        gia_bq=("close", "mean"),
        he_so_bien_thien=("close", bien_do_trung_binh),
        so_phien=("close", "count"),
    )
    .query("so_phien > 400")
    .nlargest(8, "he_so_bien_thien")
)
print("8 mã có giá biến động mạnh nhất quanh trung bình của chính nó:")
bien_thien.round(2)

8 mã có giá biến động mạnh nhất quanh trung bình của chính nó:


,gia_bq,he_so_bien_thien,so_phien
symbol,,,
GEE,62.32,59.78,497
GEX,23.76,37.68,499
BSR,17.27,37.42,491
CDC,13.03,36.63,499
DCL,32.37,31.44,499
CKG,13.07,31.42,499
CII,16.30,30.38,499
ASP,5.30,30.33,499


## 3 · `transform` — công cụ bị đánh giá thấp nhất

`transform` trả về **cùng số dòng với đầu vào**, nên nó gán thẳng vào frame
được. Đây là cách đúng để thêm cột "so với nhóm của chính nó".

In [6]:
co_chuan = gia.assign(
    gia_bq_ma=lambda d: d.groupby("symbol", observed=True)["close"].transform("mean"),
    lech_so_bq=lambda d: (d["close"] / d["gia_bq_ma"] - 1) * 100,
    kl_bq_ma=lambda d: d.groupby("symbol", observed=True)["volume"].transform("mean"),
    so_lan_kl=lambda d: d["volume"] / d["kl_bq_ma"],
)

print(f"Frame vẫn {len(co_chuan):,} dòng — transform không gộp nhóm")
co_chuan[["symbol", "date", "close", "gia_bq_ma", "lech_so_bq", "so_lan_kl"]].head(4).round(
    {"close": 2, "gia_bq_ma": 2, "lech_so_bq": 2, "so_lan_kl": 2}
)

Frame vẫn 73,054 dòng — transform không gộp nhóm


,symbol,date,close,gia_bq_ma,lech_so_bq,so_lan_kl
0,AAA,2024-08-12,9.61,7.59,26.68,2.24
1,AAA,2024-08-13,9.66,7.59,27.34,1.50
2,AAA,2024-08-14,9.57,7.59,26.15,1.55
3,AAA,2024-08-15,9.34,7.59,23.12,2.11


### Cùng việc đó bằng `agg` + `merge` — dài hơn và dễ sai hơn

In [7]:
# Cách dài
tam = gia.groupby("symbol", observed=True)["close"].mean().rename("gia_bq_ma")
cach_dai = gia.merge(tam, on="symbol", how="left")

# Cách ngắn
cach_ngan = gia.assign(
    gia_bq_ma=lambda d: d.groupby("symbol", observed=True)["close"].transform("mean")
)

print(f"merge     : {cach_dai.shape}")
print(f"transform : {cach_ngan.shape}")
print(f"Cùng kết quả: {np.allclose(cach_dai['gia_bq_ma'], cach_ngan['gia_bq_ma'])}")
print()
print("→ transform không tạo frame trung gian, không đổi thứ tự dòng, không mất index.")

merge     : (73054, 8)


transform : (73054, 8)
Cùng kết quả: True

→ transform không tạo frame trung gian, không đổi thứ tự dòng, không mất index.


### `transform` với hàm cửa sổ trượt

Đây là chỗ nó không thay thế được: trung bình động **trong từng mã**.

In [8]:
ma_dong = gia.assign(
    ma20=lambda d: d.groupby("symbol", observed=True)["close"].transform(
        lambda s: s.rolling(20, min_periods=20).mean()
    ),
    tren_ma20=lambda d: d["close"] > d["ma20"],
)

phien_cuoi = ma_dong["date"].max()
hom_nay_bang = ma_dong[ma_dong["date"] == phien_cuoi]
print(f"Phiên {phien_cuoi:%d/%m/%Y}: {hom_nay_bang['tren_ma20'].sum()}/{len(hom_nay_bang)} mã trên MA20")

# Kiểm chứng ranh giới nhóm
dau_ma = ma_dong.groupby("symbol", observed=True).head(19)
print(f"19 dòng đầu mỗi mã có ma20 = NaN: {dau_ma['ma20'].isna().all()}")

Phiên 12/08/2026: 108/148 mã trên MA20


19 dòng đầu mỗi mã có ma20 = NaN: True


## 4 · `filter` — giữ hoặc bỏ **cả nhóm**

Khác `query`: `query` lọc từng dòng, `filter` lọc từng nhóm.

In [9]:
du_lich_su = gia.groupby("symbol", observed=True).filter(lambda g: len(g) >= 480)
thanh_khoan = gia.groupby("symbol", observed=True).filter(
    lambda g: (g["close"] * g["volume"] * 1_000).mean() >= 10e9
)

print(f"Gốc                      : {gia['symbol'].nunique()} mã")
print(f"filter đủ 480 phiên      : {du_lich_su['symbol'].nunique()} mã")
print(f"filter GTGD ≥ 10 tỷ/phiên: {thanh_khoan['symbol'].nunique()} mã")

Gốc                      : 149 mã


filter đủ 480 phiên      : 145 mã
filter GTGD ≥ 10 tỷ/phiên: 59 mã


⚠️ `filter` gọi hàm Python **một lần cho mỗi nhóm**, nên với vài trăm nhóm thì
ổn, với hàng chục nghìn nhóm thì chậm. Cách nhanh hơn là tính điều kiện bằng
`agg` rồi lọc bằng `isin`:

In [10]:
t0 = time.perf_counter()
_ = gia.groupby("symbol", observed=True).filter(
    lambda g: (g["close"] * g["volume"] * 1_000).mean() >= 10e9
)
t_filter = time.perf_counter() - t0

t0 = time.perf_counter()
dat = (
    gia.assign(gtgd=lambda d: d["close"] * d["volume"] * 1_000)
    .groupby("symbol", observed=True)["gtgd"]
    .mean()
    .pipe(lambda s: s[s >= 10e9].index)
)
_ = gia[gia["symbol"].isin(dat)]
t_agg = time.perf_counter() - t0

print(f"filter(lambda) : {t_filter * 1000:>7.1f} ms")
print(f"agg + isin     : {t_agg * 1000:>7.1f} ms   ← nhanh gấp {t_filter / t_agg:.1f} lần")

filter(lambda) :    64.3 ms


agg + isin     :    12.6 ms   ← nhanh gấp 5.1 lần


## 5 · ⚠️ pandas 3.0 đổi mặc định `observed=`

Khi cột nhóm là `category`, `groupby` phải quyết định: có trả về cả những hạng
mục **không xuất hiện** trong dữ liệu không?

**pandas 2.x mặc định `observed=False`** (trả về cả hạng mục rỗng).
**pandas 3.0 mặc định `observed=True`** (chỉ hạng mục có mặt).

In [11]:
mau = pd.DataFrame(
    {
        "nganh": pd.Categorical(
            ["Ngân hàng", "Ngân hàng", "Thép"],
            categories=["Ngân hàng", "Thép", "Bảo hiểm", "Dầu khí"],
        ),
        "gia": [20.0, 21.0, 30.0],
    }
)
print(f"Cột 'nganh' khai {len(mau['nganh'].cat.categories)} hạng mục, dữ liệu chỉ có {mau['nganh'].nunique()}")
print()
print(f"groupby mặc định (3.0) → {len(mau.groupby('nganh')['gia'].sum())} nhóm")
print(f"observed=False         → {len(mau.groupby('nganh', observed=False)['gia'].sum())} nhóm")
print(f"observed=True          → {len(mau.groupby('nganh', observed=True)['gia'].sum())} nhóm")

Cột 'nganh' khai 4 hạng mục, dữ liệu chỉ có 2

groupby mặc định (3.0) → 2 nhóm
observed=False         → 4 nhóm
observed=True          → 2 nhóm


In [12]:
print("observed=False cho ra các nhóm rỗng với giá trị 0:")
print(mau.groupby("nganh", observed=False)["gia"].sum().to_string())

observed=False cho ra các nhóm rỗng với giá trị 0:
nganh
Ngân hàng    41.0
Thép         30.0
Bảo hiểm      0.0
Dầu khí       0.0


⚠️ **Hai hệ quả trái ngược nhau, tuỳ bạn đang làm gì.**

Nếu bạn muốn một bảng có **đủ mọi hạng mục** — ví dụ báo cáo phải liệt kê cả 19
ngành ICB kể cả ngành hôm nay không có mã nào giao dịch — thì code pandas 2.x
của bạn đang dựa vào `observed=False`, và ở pandas 3.0 nó **âm thầm mất dòng**.

Ngược lại, nếu bạn từng ngạc nhiên vì `groupby` cho ra hàng nghìn nhóm rỗng
(tích Descartes của nhiều cột category), thì pandas 3.0 vừa sửa nó cho bạn.

**Cách an toàn: luôn viết rõ `observed=`.** Đó là lý do mọi notebook trong repo
này đều có `observed=True`.

In [13]:
# Nhóm theo HAI cột category — chỗ observed=False phát nổ
hai_cot = pd.DataFrame(
    {
        "san": pd.Categorical(["HOSE"] * 3, categories=["HOSE", "HNX", "UPCOM"]),
        "nganh": pd.Categorical(
            ["Ngân hàng", "Thép", "Thép"], categories=["Ngân hàng", "Thép", "Bảo hiểm", "Dầu khí"]
        ),
        "v": [1.0, 2.0, 3.0],
    }
)
print(f"Dữ liệu có {len(hai_cot)} dòng, {hai_cot.groupby(['san', 'nganh'], observed=True).ngroups} tổ hợp thật")
print(f"observed=False → {hai_cot.groupby(['san', 'nganh'], observed=False).ngroups} nhóm "
      f"(3 sàn × 4 ngành, phần lớn rỗng)")

Dữ liệu có 3 dòng, 2 tổ hợp thật
observed=False → 12 nhóm (3 sàn × 4 ngành, phần lớn rỗng)


## 6 · ⚠️ Bẫy ranh giới nhóm — lần thứ ba trong repo này

Notebook `31` đo nó với TA-Lib, notebook `54` đo với `pct_change`. Đây là dạng
tổng quát: **mọi phép nhìn sang dòng bên cạnh đều phải đi qua `groupby`.**

In [14]:
sap_xep = gia.sort_values(["symbol", "date"])

dung = sap_xep.assign(
    ls=lambda d: d.groupby("symbol", observed=True)["close"].pct_change() * 100,
    gia_truoc=lambda d: d.groupby("symbol", observed=True)["close"].shift(1),
    dinh_20=lambda d: d.groupby("symbol", observed=True)["close"].transform(
        lambda s: s.rolling(20, closed="left").max()
    ),
)
sai = sap_xep.assign(
    ls=lambda d: d["close"].pct_change() * 100,
    gia_truoc=lambda d: d["close"].shift(1),
    dinh_20=lambda d: d["close"].rolling(20, closed="left").max(),
)

dau_moi_ma = dung.groupby("symbol", observed=True).head(1).index
so_sanh = pd.DataFrame(
    {
        "đúng — NaN ở dòng đầu mỗi mã": [dung.loc[dau_moi_ma, c].isna().sum() for c in ("ls", "gia_truoc", "dinh_20")],
        "sai — NaN ở dòng đầu mỗi mã": [sai.loc[dau_moi_ma, c].isna().sum() for c in ("ls", "gia_truoc", "dinh_20")],
    },
    index=["ls", "gia_truoc", "dinh_20"],
)
so_sanh["số mã"] = len(dau_moi_ma)
so_sanh

,đúng — NaN ở dòng đầu mỗi mã,sai — NaN ở dòng đầu mỗi mã,số mã
ls,149,1,149
gia_truoc,149,1,149
dinh_20,149,1,149


In [15]:
khac = ~np.isclose(dung["ls"], sai["ls"], equal_nan=True)  # NaN vs số PHẢI tính là khác
print(f"Số dòng khác nhau giữa hai cách: {khac.sum():,} / {len(gia):,}")
print(f"Lợi suất bịa lớn nhất do cách sai: {sai.loc[dau_moi_ma, 'ls'].abs().max():,.0f}%")
print()
print("Nhìn cụ thể chỗ chuyển từ mã này sang mã kia:")
chuyen = dau_moi_ma[5]
print(
    sap_xep.loc[chuyen - 1 : chuyen + 1, ["symbol", "date", "close"]]
    .assign(ls_dung=dung.loc[chuyen - 1 : chuyen + 1, "ls"].round(2),
            ls_sai=sai.loc[chuyen - 1 : chuyen + 1, "ls"].round(2))
    .to_string(index=False)
)

Số dòng khác nhau giữa hai cách: 148 / 73,054
Lợi suất bịa lớn nhất do cách sai: 2,881%

Nhìn cụ thể chỗ chuyển từ mã này sang mã kia:
symbol       date  close  ls_dung  ls_sai
   ABR 2026-08-12  13.50     6.72    6.72
   ABS 2024-08-12   4.30      NaN  -68.15
   ABS 2024-08-13   4.27    -0.70   -0.70


## 7 · Hiệu năng: `category` làm `groupby` nhanh hơn

Notebook `51` đo `category` tiết kiệm bộ nhớ. Đây là mặt thứ hai của nó.

In [16]:
dang_str = gia.copy()
dang_cat = gia.astype({"symbol": "category"})


def do_groupby(df: pd.DataFrame, lan: int = 20) -> float:
    t0 = time.perf_counter()
    for _ in range(lan):
        df.groupby("symbol", observed=True)["close"].mean()
    return (time.perf_counter() - t0) / lan * 1000


t_str = do_groupby(dang_str)
t_cat = do_groupby(dang_cat)

print(f"groupby trên cột 'str'      : {t_str:>6.1f} ms")
print(f"groupby trên cột 'category' : {t_cat:>6.1f} ms   ← nhanh gấp {t_str / t_cat:.1f} lần")
print()
print(f"Bộ nhớ cột symbol: {dang_str['symbol'].memory_usage(deep=True) / 1024:,.0f} KB "
      f"→ {dang_cat['symbol'].memory_usage(deep=True) / 1024:,.0f} KB")

groupby trên cột 'str'      :    5.7 ms
groupby trên cột 'category' :    2.0 ms   ← nhanh gấp 2.8 lần



Bộ nhớ cột symbol: 3,710 KB → 150 KB


`category` đã mã hoá mỗi mã thành một số nguyên, nên `groupby` chỉ việc nhóm
theo số thay vì so sánh chuỗi. Với dữ liệu dạng long nhiều mã, đây là phép tối
ưu một dòng đáng làm nhất.

## 8 · `apply` — dùng khi ba cái kia không đủ

`groupby.apply` nhận **cả frame của nhóm**, nên nó làm được thứ `agg` và
`transform` không làm được: trả về nhiều dòng, nhiều cột, hình dạng tuỳ ý.

In [17]:
def ba_phien_manh_nhat(g: pd.DataFrame) -> pd.DataFrame:
    """Ba phiên tăng mạnh nhất của một mã — nhiều dòng, nhiều cột."""
    return g.assign(ls=g["close"].pct_change() * 100).nlargest(3, "ls")[["date", "close", "ls"]]


BA_MA = gia["symbol"].unique()[:3].tolist()  # lấy từ chính frame, đừng viết cứng
manh_nhat = (
    gia[gia["symbol"].isin(BA_MA)]
    .groupby("symbol", observed=True)
    .apply(ba_phien_manh_nhat)
)
print(f"Ba mã lấy từ frame: {BA_MA}")
print(f"Kết quả: {manh_nhat.shape} — 3 mã × 3 phiên")
manh_nhat.round({"close": 2, "ls": 2})

Ba mã lấy từ frame: ['AAA', 'AAM', 'AAN']
Kết quả: (9, 3) — 3 mã × 3 phiên


date  close    ls
symbol                             
AAA    240  2025-07-30   7.99  6.96
       164  2025-04-10   6.30  6.78
       319  2025-11-20   8.32  5.99
AAM    667  2025-04-16   6.63  7.11
       926  2026-05-05   6.62  6.95
       663  2025-04-10   6.58  6.82
AAN    1000 2026-05-27  15.57  6.86
       1001 2026-05-28  16.63  6.81
       999  2026-05-26  14.57  6.74

⚠️ **Ba điều về `groupby.apply`:**

1. Nó **chậm** — gọi Python một lần mỗi nhóm, giống `filter`.
2. Ở pandas 3.0, hàm nhận vào **không còn chứa cột nhóm** (`include_groups`
   mặc định đã đổi). Code cũ đọc `g["symbol"]` bên trong sẽ ném `KeyError`.
3. Kết quả có `MultiIndex` — cột nhóm thành tầng ngoài của index.

In [18]:
def doc_cot_nhom(g: pd.DataFrame):
    return g["symbol"].iloc[0]


try:
    gia.groupby("symbol", observed=True).head(50).groupby("symbol", observed=True).apply(doc_cot_nhom)
except KeyError as e:
    print(f"Đọc cột nhóm bên trong apply → KeyError: {e}")
print()
print("Cách đúng: dùng `g.name` để biết mình đang ở nhóm nào")
print(
    gia.groupby("symbol", observed=True)
    .head(50)
    .groupby("symbol", observed=True)
    .apply(lambda g: f"{g.name}: {len(g)} dòng")
    .head(4)
    .to_string()
)

Đọc cột nhóm bên trong apply → KeyError: 'symbol'

Cách đúng: dùng `g.name` để biết mình đang ở nhóm nào
symbol
AAA    AAA: 50 dòng
AAM    AAM: 50 dòng
AAN    AAN: 50 dòng
AAT    AAT: 50 dòng


## 9 · Ghép lại: bảng xếp hạng ngành

Một biểu thức, dùng cả `agg` lẫn `transform`.

In [19]:
xep_hang = (
    gia.merge(danh_muc[["symbol", "icb_name2"]], on="symbol")
    .sort_values(["symbol", "date"])
    .assign(
        ls=lambda d: d.groupby("symbol", observed=True)["close"].pct_change() * 100,
        gtgd=lambda d: d["close"] * d["volume"] * 1_000,
    )
    .groupby("icb_name2", observed=True)
    .agg(
        so_ma=("symbol", "nunique"),
        ls_bq_phien=("ls", "mean"),
        do_bien_dong=("ls", "std"),
        gtgd_ty_phien=("gtgd", "mean"),
    )
    .assign(gtgd_ty_phien=lambda d: (d["gtgd_ty_phien"] / 1e9).round(1))
    .query("so_ma >= 5")
    .sort_values("ls_bq_phien", ascending=False)
)
print(f"{len(xep_hang)} ngành có từ 5 mã trở lên")
xep_hang.round(3)

12 ngành có từ 5 mã trở lên


,so_ma,ls_bq_phien,do_bien_dong,gtgd_ty_phien
icb_name2,,,,
Hàng & Dịch vụ Công nghiệp,10,0.078,2.219,62.5
Ngân hàng,6,0.066,1.932,249.2
Bán lẻ,7,0.053,2.239,23.5
Thực phẩm và đồ uống,15,0.043,2.132,29.2
Xây dựng và Vật liệu,26,0.031,2.243,23.4
Y tế,6,0.029,1.768,6.9
Tài nguyên Cơ bản,8,0.019,1.991,2.8
"Điện, nước & xăng dầu khí đốt",11,0.017,1.844,13.5
Hóa chất,12,-0.006,2.157,54.7


## Tổng kết

| Bạn cần | Dùng |
|---|---|
| Một dòng mỗi nhóm | `.agg(ten=("cot", "ham"))` — named aggregation |
| Thêm cột so với nhóm | `.transform("mean")` — giữ nguyên hình dạng |
| Giữ/bỏ cả nhóm | `.filter(lambda g: …)` — hoặc `agg` + `isin` khi nhiều nhóm |
| Hình dạng tuỳ ý | `.apply(...)` — chậm, dùng cuối cùng |
| Biết đang ở nhóm nào trong `apply` | `g.name` |

**Bốn điều mang sang notebook sau:**

1. ⚠️ **pandas 3.0 đổi mặc định `observed=` thành `True`** cho cột `category`.
   Báo cáo cần đủ mọi hạng mục sẽ âm thầm mất dòng. **Luôn viết rõ `observed=`.**
2. **`transform` thay được `agg` + `merge`** trong phần lớn trường hợp — ngắn
   hơn, không frame trung gian, không đổi thứ tự dòng.
3. **Mọi phép nhìn sang dòng bên cạnh phải qua `groupby`** — `shift`,
   `pct_change`, `rolling`, `diff`, `cumsum`.
4. **`groupby` trên `category` nhanh hơn trên `str`** và tốn ít bộ nhớ hơn — một
   dòng `astype` trả công ở mọi bước sau.

---

**Tiếp theo:** [`56_reshape_long_wide.ipynb`](56_reshape_long_wide.ipynb) —
`pivot`, `melt`, `stack`, `unstack`, và vì sao dữ liệu finlens ở dạng long.